In [19]:
# --- Step 1: Import Libraries ---
import pandas as pd
import numpy as np

# --- Step 2: Load Dataset ---
# Make sure you have uploaded "OnlineRetail.csv"
data = pd.read_csv('OnlineRetail.csv', encoding='ISO-8859-1')

# --- Step 3: Data Cleaning ---
data.dropna(inplace=True)                                       # remove nulls
data = data[~data['InvoiceNo'].astype(str).str.startswith('C')] # remove cancelled invoices
data = data[data['Quantity'] > 0]                               # keep only positive quantity

# ⚡ Optional: Work with a subset for speed (e.g., first 5000 invoices)
subset_invoices = data['InvoiceNo'].unique()[:5000]
data = data[data['InvoiceNo'].isin(subset_invoices)]

# --- Step 4: Prepare Transaction Dataset ---
transactions = (data.groupby("InvoiceNo")['Description']
                   .apply(list)
                   .tolist())

print("Sample Transaction:", transactions[0][:10])  # show first 10 items of first invoice


# --- Step 5: Support Calculation Function with Cache ---
support_cache = {}

def get_support(itemset, transactions):
    """
    Calculate support of an itemset with caching.
    """
    key = tuple(sorted(itemset))
    if key in support_cache:
        return support_cache[key]
    
    count = 0
    for trans in transactions:
        if set(itemset).issubset(set(trans)):
            count += 1
    sup = count / len(transactions)
    support_cache[key] = sup
    return sup


# --- Step 6: Candidate Generation ---
def generate_candidates(prev_frequent_itemsets, k):
    """
    Generate candidate itemsets of size k from frequent itemsets of size (k-1).
    """
    candidates = []
    prev_items = list(prev_frequent_itemsets.keys())

    for i in range(len(prev_items)):
        for j in range(i+1, len(prev_items)):
            union_set = tuple(sorted(set(prev_items[i]) | set(prev_items[j])))
            if len(union_set) == k and union_set not in candidates:
                candidates.append(union_set)
    return candidates


# --- Step 7: Apriori Algorithm ---
def apriori(transactions, min_support=0.05):
    """
    Apriori algorithm implemented from scratch with caching.
    Returns dict of frequent itemsets with their support.
    """
    num_trans = len(transactions)
    freq_itemsets = {}

    # Step 1: Get frequent 1-itemsets
    items = set(i for trans in transactions for i in trans)
    L1 = {}
    for item in items:
        sup = get_support([item], transactions)
        if sup >= min_support:
            L1[(item,)] = sup

    all_freq_itemsets = dict(L1)
    k = 2
    current_frequents = L1

    while current_frequents:
        # Step 2: Generate candidates of size k
        candidates = generate_candidates(current_frequents, k)

        Lk = {}
        for cand in candidates:
            sup = get_support(cand, transactions)
            if sup >= min_support:
                Lk[cand] = sup

        all_freq_itemsets.update(Lk)
        current_frequents = Lk
        k += 1

    return all_freq_itemsets


# --- Step 8: Run Apriori ---
frequent_itemsets = apriori(transactions, min_support=0.05)

print("\nTop Frequent Itemsets:")
for item, sup in sorted(frequent_itemsets.items(), key=lambda x: -x[1])[:10]:
    print(item, ":", round(sup, 4))


# --- Step 9: Association Rule Generation ---
def generate_rules(frequent_itemsets, transactions, min_conf=0.5):
    """
    Generate association rules from frequent itemsets.
    """
    rules = []
    for itemset, sup in frequent_itemsets.items():
        if len(itemset) < 2:
            continue
        for i in range(len(itemset)):
            antecedent = (itemset[i],)
            consequent = tuple(sorted(set(itemset) - set(antecedent)))

            support_ab = sup
            support_a = get_support(antecedent, transactions)
            support_b = get_support(consequent, transactions)

            confidence = support_ab / support_a if support_a > 0 else 0
            lift = confidence / support_b if support_b > 0 else 0

            if confidence >= min_conf:
                rules.append({
                    "antecedent": antecedent,
                    "consequent": consequent,
                    "support": round(support_ab, 4),
                    "confidence": round(confidence, 4),
                    "lift": round(lift, 4)
                })
    return pd.DataFrame(rules)


rules_df = generate_rules(frequent_itemsets, transactions, min_conf=0.5)

print("\nTop Association Rules:")
print(rules_df.head())


Sample Transaction: ['WHITE HANGING HEART T-LIGHT HOLDER', 'WHITE METAL LANTERN', 'CREAM CUPID HEARTS COAT HANGER', 'KNITTED UNION FLAG HOT WATER BOTTLE', 'RED WOOLLY HOTTIE WHITE HEART.', 'SET 7 BABUSHKA NESTING BOXES', 'GLASS STAR FROSTED T-LIGHT HOLDER']

Top Frequent Itemsets:
('WHITE HANGING HEART T-LIGHT HOLDER',) : 0.1382
('REGENCY CAKESTAND 3 TIER',) : 0.1148
('SET OF 3 CAKE TINS PANTRY DESIGN ',) : 0.0946
('JUMBO BAG RED RETROSPOT',) : 0.0778
('JAM MAKING SET PRINTED',) : 0.072
('ASSORTED COLOUR BIRD ORNAMENT',) : 0.0718
('JAM MAKING SET WITH JARS',) : 0.0686
('PACK OF 72 RETROSPOT CAKE CASES',) : 0.067
('HEART OF WICKER SMALL',) : 0.0668
('NATURAL SLATE HEART CHALKBOARD ',) : 0.066

Top Association Rules:
Empty DataFrame
Columns: []
Index: []


In [11]:
# =========================
# Online Retail: Fast Market Basket Analysis
# Toggle between FP-Growth (fast) and Apriori
# =========================

# If running in a fresh environment, uncomment the next line:
# !pip -q install mlxtend

import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
from time import time

# --------- Parameters ---------
DATA_FILE       = "OnlineRetail.csv"   # your CSV file
ENCODING        = "ISO-8859-1"
COUNTRY_FILTER  = None                 # e.g., "United Kingdom" or None for all
MIN_SUPPORT     = 0.02                 # raise for speed, lower to find rarer patterns
MIN_CONFIDENCE  = 0.5
MIN_LIFT        = 1.0
MAX_LEN         = 3                    # 2 or 3 keeps things fast
USE_FPGROWTH    = True                 # True = FP-Growth (faster), False = Apriori
SAVE_PREFIX     = "retail"

t0 = time()

# --------- Load & Clean ---------
df = pd.read_csv(DATA_FILE, encoding=ENCODING)

# Standard cleanups
df = df.dropna(subset=["InvoiceNo", "Description", "Quantity", "UnitPrice", "Country"])
df["InvoiceNo"] = df["InvoiceNo"].astype(str)
df = df[~df["InvoiceNo"].str.startswith("C")]           # remove cancellations
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]   # positive quantity & price

# Normalize descriptions (helps deduplicate near-identical strings)
df["Description"] = (df["Description"]
                     .str.strip()
                     .str.replace(r"\s+", " ", regex=True))

if COUNTRY_FILTER:
    df = df[df["Country"] == COUNTRY_FILTER]

# --------- Prune Rare Items BEFORE pivot (huge speed boost) ---------
# Compute in how many distinct invoices each item appears
inv_per_item = (df.groupby("Description")["InvoiceNo"]
                  .nunique()
                  .sort_values(ascending=False))

num_invoices = df["InvoiceNo"].nunique()
min_item_tx  = max(int(np.ceil(MIN_SUPPORT * num_invoices)), 2)  # at least 2 transactions
popular_items = inv_per_item[inv_per_item >= min_item_tx].index

df = df[df["Description"].isin(popular_items)]

# --------- Build Basket (Invoice x Items, boolean) ---------
# Use crosstab which is memory-efficient for this shape
basket = pd.crosstab(df["InvoiceNo"], df["Description"])
basket = (basket > 0)  # boolean matrix

print(f"Invoices: {basket.shape[0]:,} | Items (after prune): {basket.shape[1]:,}")

# --------- Frequent Itemsets ---------
algo_t0 = time()
if USE_FPGROWTH:
    frequent_itemsets = fpgrowth(
        basket,
        min_support=MIN_SUPPORT,
        use_colnames=True,
        max_len=MAX_LEN
    )
else:
    frequent_itemsets = apriori(
        basket,
        min_support=MIN_SUPPORT,
        use_colnames=True,
        max_len=MAX_LEN
    )
algo_t1 = time()

# Sort by support (desc)
frequent_itemsets = frequent_itemsets.sort_values("support", ascending=False).reset_index(drop=True)

print(f"Found {len(frequent_itemsets):,} frequent itemsets in {algo_t1 - algo_t0:.2f}s.")
print(frequent_itemsets.head(10))

# --------- Association Rules ---------
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=MIN_CONFIDENCE
)

# Optional: filter by lift as well
rules = rules[rules["lift"] >= MIN_LIFT].copy()

# Neaten columns
def fs_to_tuple(fs):
    return tuple(sorted(list(fs)))

rules["antecedent"] = rules["antecedents"].apply(fs_to_tuple)
rules["consequent"] = rules["consequents"].apply(fs_to_tuple)

keep_cols = [
    "antecedent", "consequent", "support", "confidence", "lift",
    "leverage", "conviction"
]
rules = (rules[keep_cols]
         .sort_values(["lift", "confidence", "support"], ascending=False)
         .reset_index(drop=True))

print("\nTop rules:")
print(rules.head(10))

# --------- Save Outputs ---------
fi_path = f"{SAVE_PREFIX}_frequent_itemsets.csv"
rl_path = f"{SAVE_PREFIX}_association_rules.csv"
frequent_itemsets.to_csv(fi_path, index=False)
rules.to_csv(rl_path, index=False)

print(f"\nSaved:\n- {fi_path}\n- {rl_path}")
print(f"Total runtime: {time() - t0:.2f}s")


Invoices: 17,976 | Items (after prune): 297
Found 421 frequent itemsets in 12.10s.
    support                              itemsets
0  0.125501  (WHITE HANGING HEART T-LIGHT HOLDER)
1  0.116211             (JUMBO BAG RED RETROSPOT)
2  0.110592            (REGENCY CAKESTAND 3 TIER)
3  0.093736                       (PARTY BUNTING)
4  0.087005             (LUNCH BAG RED RETROSPOT)
5  0.080941       (ASSORTED COLOUR BIRD ORNAMENT)
6  0.077047    (SET OF 3 CAKE TINS PANTRY DESIGN)
7  0.073431     (PACK OF 72 RETROSPOT CAKE CASES)
8  0.071429               (LUNCH BAG SUKI DESIGN)
9  0.070817              (LUNCH BAG BLACK SKULL.)

Top rules:
                                          antecedent  \
0              (WOODEN STAR CHRISTMAS SCANDINAVIAN,)   
1             (WOODEN HEART CHRISTMAS SCANDINAVIAN,)   
2                  (PINK REGENCY TEACUP AND SAUCER,)   
3  (GREEN REGENCY TEACUP AND SAUCER, ROSES REGENC...   
4  (PINK REGENCY TEACUP AND SAUCER, ROSES REGENCY...   
5                 (